In [ ]:
import os, sys

# GPU 디바이스
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# 경로
HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"

os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# logger Jupyter 호환 패치
import logging
import utils.logger as _log_mod

def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S")
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(fmt)
    logger.addHandler(h)
    return logger

_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
# 의존성 설치 (최초 1회만 실행)
!pip install pytorch-forecasting pytorch-lightning --quiet

In [ ]:
"""
TFT (Temporal Fusion Transformer) 주가 예측 모델 학습
- pytorch-forecasting 기반
- GPU 가속 학습
"""
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# lightning 호환: pytorch-forecasting v1.x는 lightning.pytorch 사용
try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import CrossEntropy
from pytorch_forecasting.data import NaNLabelEncoder

warnings.filterwarnings("ignore")
pl.seed_everything(42)

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {pl.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ========== 설정 ==========
BASE_PATH = Path(os.environ.get("BASE_PATH", "G:/내 드라이브/kospi200-project"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "tft"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# 학습 설정
TRAIN_END = "2024-06-30"
VAL_START = "2024-07-01"
VAL_END = "2024-12-31"
TEST_START = "2025-01-01"

# TFT 하이퍼파라미터
MAX_ENCODER_LENGTH = 30          # 과거 참조 기간 (거래일)
MAX_PREDICTION_LENGTH = 1        # 예측 기간
BATCH_SIZE = 128
MAX_EPOCHS = 50
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
ATTENTION_HEAD_SIZE = 4
DROPOUT = 0.3
HIDDEN_CONTINUOUS_SIZE = 32
GRADIENT_CLIP_VAL = 0.1

print("설정 완료")
print(f"  데이터: {BASE_PATH}")
print(f"  학습: ~ {TRAIN_END}")
print(f"  검증: {VAL_START} ~ {VAL_END}")
print(f"  테스트: {TEST_START} ~")
print(f"  인코더 길이: {MAX_ENCODER_LENGTH}일")

In [ ]:
# ========== 데이터 로드 ==========
print("TFT 피처 로드 중...")
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])

# sector_id를 string으로 변환 (categorical용)
df["sector_id"] = df["sector_id"].astype(str)
# target은 정수 클래스 라벨 (CrossEntropy용)
df["target_5d"] = df["target_5d"].astype(int)

print(f"  전체: {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"  기간: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"  피처: {len(df.columns)} columns")
print(f"  타겟 분포: {df['target_5d'].value_counts().to_dict()}")

# NaN 확인
nan_cols = df.isnull().sum()
nan_cols = nan_cols[nan_cols > 0]
if len(nan_cols) > 0:
    print(f"\n  NaN 컬럼: {dict(nan_cols)}")
else:
    print("\n  NaN 없음")

In [ ]:
# ========== 시간 기반 분할 ==========
train_df = df[df["date"] <= TRAIN_END].copy()
val_df = df[(df["date"] >= VAL_START) & (df["date"] <= VAL_END)].copy()
test_df = df[df["date"] >= TEST_START].copy()

print(f"학습: {len(train_df):,} rows ({train_df['date'].min().date()} ~ {train_df['date'].max().date()})")
print(f"검증: {len(val_df):,} rows ({val_df['date'].min().date()} ~ {val_df['date'].max().date()})")
print(f"테스트: {len(test_df):,} rows ({test_df['date'].min().date()} ~ {test_df['date'].max().date()})")

# 학습+검증 합침 (TimeSeriesDataSet이 내부적으로 분할)
train_val_df = df[df["date"] <= VAL_END].copy()
train_max_time_idx = train_df["time_idx"].max()
print(f"\n학습 time_idx 범위: 0 ~ {train_max_time_idx}")

In [ ]:
# ========== TimeSeriesDataSet 정의 ==========

# 시변 관측 피처 (과거만 사용 가능)
time_varying_known = [
    "day_of_week", "month", "week_of_year",
    "is_month_end", "is_quarter_end",
]

time_varying_observed = [
    "daily_return", "log_volume_change", "high_low_range",
    "rsi_14", "macd_norm", "bb_percent_b",
    "volume_ratio_5d", "momentum_5d", "momentum_20d",
    "foreign_net_ratio", "inst_net_ratio",
    "kospi200_return", "vix", "vix_change", "usd_krw_change",
    "relative_return",
]

# 실제 데이터에 있는 컬럼만 필터링
time_varying_observed = [c for c in time_varying_observed if c in df.columns]
time_varying_known = [c for c in time_varying_known if c in df.columns]
print(f"Time-varying observed: {len(time_varying_observed)} features")
print(f"Time-varying known: {len(time_varying_known)} features")

# 공통 파라미터
dataset_params = dict(
    time_idx="time_idx",
    target="target_5d",
    group_ids=["ticker"],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=["sector_id"],
    time_varying_known_reals=time_varying_known,
    time_varying_unknown_reals=time_varying_observed,
    target_normalizer=NaNLabelEncoder(),
    add_relative_time_idx=True,
    add_target_scales=False,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

# 학습 데이터셋
training = TimeSeriesDataSet(train_df, **dataset_params)

# 검증 데이터셋 — 학습+검증 전체를 넘기되 predict 모드로
# (encoder는 학습 기간에서, decoder는 검증 기간에서 가져옴)
validation = TimeSeriesDataSet.from_dataset(
    training,
    train_val_df,
    predict=True,
    stop_randomization=True,
)

print(f"\n학습 데이터셋: {len(training)} samples")
print(f"검증 데이터셋: {len(validation)} samples")

# DataLoader
train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)

In [ ]:
# ========== TFT 모델 정의 ==========
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTENTION_HEAD_SIZE,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
    output_size=2,            # 이진 분류: 2 클래스 (하락/상승)
    loss=CrossEntropy(),
    log_interval=10,
    reduce_on_plateau_patience=5,
    reduce_on_plateau_min_lr=1e-6,
)

print(f"모델 파라미터 수: {tft.size() / 1e3:.1f}K")
print(tft)

In [ ]:
# ========== 학습 ==========
early_stop = EarlyStopping(monitor="val_loss", patience=8, verbose=True, mode="min")
lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=GRADIENT_CLIP_VAL,
    callbacks=[early_stop, lr_monitor],
    enable_progress_bar=True,
    log_every_n_steps=50,
)

print("학습 시작...")
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
print(f"학습 완료! Best val_loss: {early_stop.best_score:.4f}")

In [ ]:
# ========== 최적 모델 로드 & 검증 예측 ==========
best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Best model: {best_model_path}")

best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# 검증 세트 예측 (output shape: [batch, prediction_length, 2])
val_raw = best_tft.predict(val_dataloader, mode="raw", return_x=True)
val_predictions = best_tft.predict(val_dataloader, return_x=True)

# 상승 확률 추출 (class 1 = 상승)
val_output = val_raw.output["prediction"].squeeze(1)  # [batch, 2]
val_probs = torch.softmax(val_output, dim=-1)[:, 1].cpu().numpy()

print(f"검증 예측 shape: {val_probs.shape}")
print(f"상승 확률 분포: mean={val_probs.mean():.4f}, std={val_probs.std():.4f}")
print(f"확률 범위: [{val_probs.min():.4f}, {val_probs.max():.4f}]")

In [ ]:
# ========== 성능 평가 ==========
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# 검증 실제 타겟
val_actuals = torch.cat([y[0] for x, y in iter(val_dataloader)]).cpu().numpy().squeeze()
val_preds_binary = (val_probs >= 0.5).astype(int)

da = accuracy_score(val_actuals, val_preds_binary)
try:
    auc = roc_auc_score(val_actuals, val_probs)
except:
    auc = float("nan")

print("=" * 60)
print("  TFT 검증 성능")
print("=" * 60)
print(f"  Direction Accuracy (DA): {da*100:.2f}%")
print(f"  AUC-ROC: {auc:.4f}")
print(f"  양성 예측 비율: {val_preds_binary.mean()*100:.1f}%")
print()
print(classification_report(val_actuals, val_preds_binary, target_names=["하락", "상승"]))
print("=" * 60)
print(f"\n비교: LSTM DA=48.9%, LightGBM DA=50.8%")

In [ ]:
# ========== 피처 중요도 (Attention 분석) ==========
import matplotlib.pyplot as plt

interpretation = best_tft.interpret_output(val_predictions.output, reduction="sum")

# 인코더 변수 중요도
enc_importance = interpretation["encoder_variables"].cpu().numpy()
feature_names = training.reals + training.flat_categoricals
if len(feature_names) == len(enc_importance):
    sorted_idx = np.argsort(enc_importance)[::-1][:15]  # Top 15
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(sorted_idx)), enc_importance[sorted_idx][::-1])
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx][::-1])
    ax.set_xlabel("Importance")
    ax.set_title("TFT Encoder Variable Importance (Top 15)")
    plt.tight_layout()
    save_path = MODEL_SAVE_PATH / "tft_variable_importance.png"
    fig.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"저장: {save_path}")
else:
    print(f"Feature names ({len(feature_names)}) != importance ({len(enc_importance)})")
    print("Encoder importance:", enc_importance)

In [ ]:
# ========== 테스트 세트 예측 ==========
test_dataset = TimeSeriesDataSet.from_dataset(
    training,
    df[df["date"] <= df["date"].max()],
    predict=True,
    stop_randomization=True,
    min_prediction_idx=df[df["date"] >= TEST_START]["time_idx"].min(),
)

test_dataloader = test_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)

# 예측
test_raw = best_tft.predict(test_dataloader, mode="raw", return_x=True)
test_output = test_raw.output["prediction"].squeeze(1)
test_probs = torch.softmax(test_output, dim=-1)[:, 1].cpu().numpy()
test_actuals = torch.cat([y[0] for x, y in iter(test_dataloader)]).cpu().numpy().squeeze()
test_preds_binary = (test_probs >= 0.5).astype(int)

test_da = accuracy_score(test_actuals, test_preds_binary)
try:
    test_auc = roc_auc_score(test_actuals, test_probs)
except:
    test_auc = float("nan")

print("=" * 60)
print("  TFT 테스트 성능 (OOS)")
print("=" * 60)
print(f"  기간: {TEST_START} ~")
print(f"  Direction Accuracy (DA): {test_da*100:.2f}%")
print(f"  AUC-ROC: {test_auc:.4f}")
print(f"  양성 예측 비율: {test_preds_binary.mean()*100:.1f}%")
print()
print(classification_report(test_actuals, test_preds_binary, target_names=["하락", "상승"]))

In [ ]:
# ========== 모델 & 예측 저장 ==========
import json
from datetime import datetime

# 모델 저장
model_path = MODEL_SAVE_PATH / "tft_best.ckpt"
trainer.save_checkpoint(str(model_path))
print(f"모델 저장: {model_path}")

# 예측 결과 저장
pred_df = pd.DataFrame({
    "prob": test_probs.flatten(),
    "actual": test_actuals.flatten(),
    "pred": test_preds_binary.flatten(),
})
pred_path = MODEL_SAVE_PATH / f"tft_predictions_{datetime.now().strftime('%Y%m%d')}.parquet"
pred_df.to_parquet(str(pred_path))
print(f"예측 저장: {pred_path}")

# 메트릭 요약
metrics = {
    "model": "TFT",
    "val_da": round(da * 100, 2),
    "val_auc": round(float(auc), 4),
    "test_da": round(test_da * 100, 2),
    "test_auc": round(float(test_auc), 4),
    "config": {
        "encoder_length": MAX_ENCODER_LENGTH,
        "hidden_size": HIDDEN_SIZE,
        "attention_heads": ATTENTION_HEAD_SIZE,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    },
    "comparison": {"lstm_da": 48.9, "lgbm_da": 50.8},
    "timestamp": datetime.now().isoformat(),
}

metrics_path = MODEL_SAVE_PATH / "tft_metrics.json"
with open(str(metrics_path), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(f"메트릭 저장: {metrics_path}")
print("\n학습 완료!")

## 결과 요약

| 모델 | DA | AUC | 비고 |
|------|-----|-----|------|
| LSTM | 48.9% | - | 현행 모델 (시퀀스 20일, 5피처) |
| LightGBM | 50.8% | - | 방향 예측 우수 (60+ 피처) |
| **TFT** | **?%** | **?** | 위 결과 참조 (30일, 16피처 + static) |

### 다음 단계
1. Walk-Forward 검증으로 robustness 확인
2. 백테스트 (PortfolioSimulator 연결)
3. TFT + LightGBM 앙상블
4. 하이퍼파라미터 튜닝 (encoder_length, hidden_size, attention_heads)